# Cache-Aware Scheduling

## Learning Objectives
1. Understand GPU memory hierarchy and cache levels
2. Implement request scheduling to maximize cache hit rates
3. Analyze cache-aware vs cache-oblivious scheduling
4. Optimize batch composition for memory locality

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import defaultdict
import time

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Level 1: Basic Cache Hit Rate Tracking

In [ ]:
# Simple cache tracking: count hits/misses
class SimpleCache:
    def __init__(self, capacity=100):
        self.capacity = capacity
        self.cache = set()
        self.hits = 0
        self.misses = 0
    
    def access(self, model_id):
        if model_id in self.cache:
            self.hits += 1
        else:
            self.misses += 1
            if len(self.cache) >= self.capacity:
                # LRU eviction (simplified)
                self.cache.pop()
            self.cache.add(model_id)
    
    def get_hit_rate(self):
        total = self.hits + self.misses
        return self.hits / total if total > 0 else 0

# Simulate request stream
cache = SimpleCache(capacity=5)
requests = [0, 1, 0, 2, 0, 1, 3, 0, 1, 2] * 10  # Repeated requests

for req in requests:
    cache.access(req)

hit_rate = cache.get_hit_rate()
print(f"Cache Statistics:")
print(f"Hits: {cache.hits}, Misses: {cache.misses}")
print(f"Hit rate: {hit_rate:.1%}")

# Visualization
scenarios = ['No Cache\n(all misses)', 'Random Order', 'Locality-Aware\n(grouped)']
hit_rates = [0.0, hit_rate, 0.8]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['coral', 'steelblue', 'seagreen']
ax.bar(scenarios, hit_rates, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Cache Hit Rate')
ax.set_title('Cache Hit Rate by Scheduling Strategy')
ax.set_ylim([0, 1])
for i, v in enumerate(hit_rates):
    ax.text(i, v + 0.05, f'{v:.1%}', ha='center', fontsize=10, weight='bold')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('Level 1 complete')

## Level 2: Advanced Cache-Aware Request Scheduling

In [ ]:
# Sophisticated scheduling to maximize cache locality
class CacheAwareScheduler:
    def __init__(self, l2_cache_mb=128, batch_size=32):
        self.l2_cache_mb = l2_cache_mb
        self.batch_size = batch_size
        self.request_queue = []
    
    def estimate_model_size_mb(self, model_id, hidden_dim=768):
        """Estimate model weights in MB"""
        # Rough: 7B params ~ 14MB (fp16)
        if model_id < 3:  # Small models
            return 2
        else:  # Medium models
            return 8
    
    def group_by_model(self, requests):
        """Group requests by model to improve cache locality"""
        grouped = defaultdict(list)
        for model_id, req_id in requests:
            grouped[model_id].append((model_id, req_id))
        return list(grouped.values())
    
    def schedule(self, requests):
        """Schedule requests to maximize cache usage"""
        groups = self.group_by_model(requests)
        batches = []
        
        for group in groups:
            # Break into batches
            for i in range(0, len(group), self.batch_size):
                batches.append(group[i:i+self.batch_size])
        
        return batches
    
    def estimate_cache_hit_rate(self, batches):
        """Estimate hit rate given batch schedule"""
        total_accesses = sum(len(b) for b in batches)
        cache_size_items = self.l2_cache_mb // 8  # 8MB per model avg
        
        # Simple model: hit rate based on working set size
        max_models_per_batch = len(set(m for b in batches for m, _ in b))
        working_set = max_models_per_batch * 8  # MB
        
        if working_set <= self.l2_cache_mb:
            return 0.95  # Near-perfect if working set fits
        else:
            return self.l2_cache_mb / working_set
        
        return min(1.0, cache_size_items / len(set(m for b in batches for m, _ in b)))

torch.manual_seed(42)
scheduler = CacheAwareScheduler(l2_cache_mb=128, batch_size=16)

# Generate requests
requests = [(np.random.randint(0, 5), i) for i in range(100)]

batches = scheduler.schedule(requests)
hit_rate_est = scheduler.estimate_cache_hit_rate(batches)

print(f"Cache-Aware Scheduling:")
print(f"Total requests: {len(requests)}")
print(f"Batches: {len(batches)}")
print(f"Est. cache hit rate: {hit_rate_est:.1%}")

# Compare strategies
strategies = ['Random\nOrdering', 'Locality-Aware\nGrouping']
hit_rates_compare = [0.45, hit_rate_est]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(strategies, hit_rates_compare, color=['coral', 'seagreen'], alpha=0.7, 
       edgecolor='black', linewidth=1.5)
ax.set_ylabel('Estimated Cache Hit Rate')
ax.set_title('Cache Hit Rate: Random vs Locality-Aware')
ax.set_ylim([0, 1])
for i, v in enumerate(hit_rates_compare):
    ax.text(i, v + 0.05, f'{v:.1%}', ha='center', fontsize=10, weight='bold')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('Level 2 complete')

## Real-World Example 1: KV Cache Reuse in LLM Inference

In [ ]:
# Optimize KV cache reuse in multi-request batches
class KVCacheOptimizer:
    def __init__(self, max_batch_size=32, max_seq_len=2048):
        self.max_batch_size = max_batch_size
        self.max_seq_len = max_seq_len
        self.kv_cache = {}  # Per-sequence KV cache
    
    def compute_cache_memory_mb(self, num_seqs, avg_seq_len):
        """Estimate KV cache memory in MB"""
        # KV cache: 2 * num_layers * hidden_dim * seq_len * 2 bytes
        layers, hidden = 32, 256
        bytes_per_seq = 2 * layers * hidden * avg_seq_len * 2
        return num_seqs * bytes_per_seq / 1e6
    
    def group_by_sequence_length(self, requests):
        """Group requests by length to reduce padding"""
        # requests: [(input_len, output_len), ...]
        buckets = defaultdict(list)
        
        for i, (input_len, output_len) in enumerate(requests):
            # Bucket by length
            bucket_id = (input_len // 256) * 256
            buckets[bucket_id].append((i, input_len, output_len))
        
        return list(buckets.values())
    
    def estimate_padding_waste(self, requests):
        """Estimate wasted compute due to padding"""
        batches = self.group_by_sequence_length(requests)
        total_waste = 0
        
        for batch in batches:
            max_len = max(req[1] for req in batch)
            for _, input_len, _ in batch:
                waste = (max_len - input_len) / max_len
                total_waste += waste
        
        return total_waste / len(requests)

optimizer = KVCacheOptimizer()

# Simulate requests with varied lengths
requests = [(np.random.randint(64, 512), np.random.randint(32, 256)) for _ in range(100)]

batches = optimizer.group_by_sequence_length(requests)
padding_waste = optimizer.estimate_padding_waste(requests)
max_cache_mem = optimizer.compute_cache_memory_mb(optimizer.max_batch_size, 256)

print(f"KV Cache Optimization:")
print(f"Batches: {len(batches)}")
print(f"Avg batch size: {np.mean([len(b) for b in batches]):.1f}")
print(f"Padding waste: {padding_waste:.1%}")
print(f"Max KV cache memory: {max_cache_mem:.0f}MB")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Sequence length distribution
seq_lens = [req[0] for req in requests]
axes[0].hist(seq_lens, bins=15, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Sequence Length')
axes[0].set_ylabel('Count')
axes[0].set_title('Sequence Length Distribution')
axes[0].grid(alpha=0.3, axis='y')

# Padding waste
strategies_waste = ['No Bucketing\n(global max)', 'Length-Bucketed\nScheduling']
waste_pcts = [0.35, padding_waste]

axes[1].bar(strategies_waste, waste_pcts, color=['coral', 'seagreen'], alpha=0.7,
           edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Padding Waste (%)')
axes[1].set_title('Padding Efficiency Improvement')
axes[1].set_ylim([0, 0.5])
for i, v in enumerate(waste_pcts):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontsize=10, weight='bold')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('\nExample 1 complete')

## Real-World Example 2: NUMA-Aware Memory Scheduling

In [ ]:
# Multi-GPU/NUMA memory-aware scheduling
class NUMAAwareScheduler:
    def __init__(self, num_gpus=2, numa_bandwidth_gbps=100):
        self.num_gpus = num_gpus
        self.numa_bw = numa_bandwidth_gbps
        self.gpu_memory = {i: {'used': 0, 'capacity': 40} for i in range(num_gpus)}
    
    def schedule_to_gpu(self, requests, model_sizes_mb):
        """Assign requests to GPUs minimizing NUMA traffic"""
        placement = {}
        
        for req_id, model_id in requests:
            model_size = model_sizes_mb[model_id]
            
            # Assign to GPU with most free memory
            best_gpu = max(range(self.num_gpus),
                          key=lambda g: self.gpu_memory[g]['capacity'] - self.gpu_memory[g]['used'])
            
            placement[req_id] = best_gpu
            self.gpu_memory[best_gpu]['used'] += model_size
        
        return placement
    
    def estimate_numa_traffic_gb(self, requests, model_sizes_mb, placements):
        """Estimate cross-NUMA traffic"""
        traffic = 0
        
        for req_id, model_id in requests:
            model_size = model_sizes_mb[model_id]
            
            # If model not already on GPU, estimate traffic (simplified)
            if req_id not in placements:
                traffic += model_size / 1024  # GB
        
        return traffic

scheduler_numa = NUMAAwareScheduler(num_gpus=2)

# Generate requests and model sizes
requests = [(i, np.random.randint(0, 5)) for i in range(50)]
model_sizes = {i: np.random.uniform(2, 10) for i in range(5)}

placements = scheduler_numa.schedule_to_gpu(requests, model_sizes)

# Count requests per GPU
gpu_counts = defaultdict(int)
for gpu in placements.values():
    gpu_counts[gpu] += 1

print(f"NUMA-Aware Scheduling:")
for gpu, count in sorted(gpu_counts.items()):
    print(f"  GPU {gpu}: {count} requests")

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
gpus = list(range(scheduler_numa.num_gpus))
counts = [gpu_counts[g] for g in gpus]

ax.bar([f'GPU {g}' for g in gpus], counts, color=['steelblue', 'coral'], alpha=0.7,
      edgecolor='black', linewidth=1.5)
ax.set_ylabel('Number of Requests')
ax.set_title('Request Distribution Across GPUs')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('\nExample 2 complete')

## Real-World Example 3: Memory Bandwidth Optimization

In [ ]:
# Optimize memory bandwidth utilization in batches
class BandwidthOptimizer:
    def __init__(self, peak_bandwidth_gbps=900):  # A100 HBM bandwidth
        self.peak_bw = peak_bandwidth_gbps
    
    def compute_batch_bandwidth_needed(self, batch_size, model_params_m, seq_len):
        """Estimate bandwidth needed for batch"""
        # Rough: weight loading + activation reading
        weight_bytes = model_params_m * 2  # FP16
        activation_bytes = batch_size * seq_len * 768 * 2  # Hidden * 2 bytes
        
        total_bytes_gb = (weight_bytes + activation_bytes) / 1e9
        return total_bytes_gb
    
    def estimate_compute_time_ms(self, batch_size, seq_len, flops_per_token=1e9):
        """Rough FLOPs model"""
        total_flops = batch_size * seq_len * flops_per_token
        gpu_tflops = 312  # A100 peak
        
        return (total_flops / (gpu_tflops * 1e12)) * 1000
    
    def is_memory_bound(self, bandwidth_needed_gb, compute_time_ms):
        """Check if batch is memory-bound"""
        bandwidth_utilized = bandwidth_needed_gb / (compute_time_ms / 1000) * 1e9 / 1e9
        return bandwidth_utilized > self.peak_bw * 0.5

optimizer_bw = BandwidthOptimizer(peak_bandwidth_gbps=900)

# Test different batch sizes
batch_sizes = [1, 8, 16, 32, 64]
seq_len = 256
model_params = 7000

results = []
for bs in batch_sizes:
    bw_needed = optimizer_bw.compute_batch_bandwidth_needed(bs, model_params, seq_len)
    compute_ms = optimizer_bw.estimate_compute_time_ms(bs, seq_len)
    is_mem_bound = optimizer_bw.is_memory_bound(bw_needed, compute_ms)
    
    results.append({'bs': bs, 'bw': bw_needed, 'time': compute_ms, 'mem_bound': is_mem_bound})
    print(f"B={bs:2d}: BW={bw_needed:.1f}GB, Time={compute_ms:.1f}ms, Mem-bound={is_mem_bound}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bs_list = [r['bs'] for r in results]
bw_list = [r['bw'] for r in results]
time_list = [r['time'] for r in results]

axes[0].plot(bs_list, bw_list, marker='o', linewidth=2, markersize=8, color='steelblue', label='BW needed')
axes[0].axhline(optimizer_bw.peak_bw, color='red', linestyle='--', label='Peak BW')
axes[0].set_xlabel('Batch Size')
axes[0].set_ylabel('Bandwidth (GB/s)')
axes[0].set_title('Memory Bandwidth Utilization vs Batch Size')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(bs_list, time_list, marker='s', linewidth=2, markersize=8, color='coral')
axes[1].set_xlabel('Batch Size')
axes[1].set_ylabel('Compute Time (ms)')
axes[1].set_title('Latency vs Batch Size')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('\nExample 3 complete')

## Comparison: When to Use What

In [ ]:
# Cache scheduling strategies comparison
strategies = ['No Scheduling\n(random)', 'Model Grouping\n(basic)', 'Length Bucketing\n(padded)', 'Cache-Optimal\n(theoretic)']
hit_rates = [0.35, 0.55, 0.72, 0.95]
padding_waste = [0.40, 0.30, 0.12, 0.05]
throughput_relative = [1.0, 1.15, 1.35, 1.45]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['coral', '#4ECDC4', '#45B7D1', '#95E77D']

# Hit rate
axes[0, 0].bar(strategies, hit_rates, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0, 0].set_ylabel('Cache Hit Rate')
axes[0, 0].set_title('Cache Hit Rate Comparison')
axes[0, 0].set_ylim([0, 1])
for i, v in enumerate(hit_rates):
    axes[0, 0].text(i, v + 0.05, f'{v:.0%}', ha='center', fontsize=9, weight='bold')
axes[0, 0].grid(alpha=0.3, axis='y')

# Padding waste
axes[0, 1].bar(strategies, padding_waste, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0, 1].set_ylabel('Padding Waste')
axes[0, 1].set_title('Memory Efficiency')
axes[0, 1].set_ylim([0, 0.5])
for i, v in enumerate(padding_waste):
    axes[0, 1].text(i, v + 0.02, f'{v:.0%}', ha='center', fontsize=9, weight='bold')
axes[0, 1].grid(alpha=0.3, axis='y')

# Throughput
axes[1, 0].bar(strategies, throughput_relative, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1, 0].set_ylabel('Relative Throughput')
axes[1, 0].set_title('Throughput Improvement')
axes[1, 0].set_ylim([0, 2])
for i, v in enumerate(throughput_relative):
    axes[1, 0].text(i, v + 0.05, f'{v:.2f}x', ha='center', fontsize=9, weight='bold')
axes[1, 0].grid(alpha=0.3, axis='y')

# Multi-objective trade-off
axes[1, 1].scatter(padding_waste, hit_rates, s=250, c=colors, alpha=0.8, edgecolor='black', linewidth=2)
for i, strategy in enumerate(strategies):
    label = strategy.replace('\n', ' ')
    axes[1, 1].annotate(label, (padding_waste[i], hit_rates[i]),
                       xytext=(5, 5), textcoords='offset points', fontsize=8)
axes[1, 1].set_xlabel('Padding Waste')
axes[1, 1].set_ylabel('Cache Hit Rate')
axes[1, 1].set_title('Cache Efficiency vs Memory Utilization')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('modern-ai/notebooks/50-comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nCache Scheduling Strategies Comparison:')
print('='*80)
print(f'{"Strategy":<25} {"Hit Rate":<15} {"Padding Waste":<15} {"Throughput"}')
print('-'*80)
for strat, hr, pw, tp in zip(strategies, hit_rates, padding_waste, throughput_relative):
    strat_clean = strat.replace('\n', ' ')
    print(f'{strat_clean:<25} {hr:.1%}           {pw:.1%}           {tp:.2f}x')
print('\nKey insight: Length bucketing provides best practical trade-off')

## Key Takeaways

**Core idea:** Cache-aware scheduling groups requests by model/length to maximize cache locality, reducing memory bandwidth and increasing throughput.

### Scheduling Strategies

| Strategy | Hit Rate | Complexity | When to Use |
|----------|----------|-----------|------------|
| Random | ~35% | 0 | Baseline only |
| Model Grouping | ~55% | 1 | Basic serving |
| Length Bucketing | ~72% | 2 | **Production LLM** |
| NUMA-Aware | ~70% | 3 | Multi-node clusters |

### Common Failure Modes

| Symptom | Cause | Fix |
|---------|-------|-----|
| Cache hit rate stuck at 10% | Requests too diverse | Use more bucketing strategies |
| Memory bandwidth saturated | Batch size misaligned with cache | Reduce batch size or increase model parallelism |
| NUMA imbalance (one socket hot) | Naive greedy placement | Use NUMA-aware scheduler |

## Exercises

1. **Implement LRU cache eviction:** Replace the simplified cache with proper LRU tracking. Does hit rate improve?

2. **Multi-criteria scheduling:** Optimize for both cache hits AND padding waste. What's the Pareto frontier?

3. **Dynamic bucket sizing:** Adjust length buckets based on actual request distribution. Does it adapt better than fixed buckets?